### Utils file containing reussable functions

In [ ]:
!git clone https://github.com/huggingface/lerobot.git
!cd lerobot && git checkout $(git rev-list -n 1 --before="2026-10-29 12:00" origin/main)
!apt-get install -y ffmpeg
!cd lerobot && pip install -e ".[train, dataset, smolvla]"

inspired from : https://colab.research.google.com/github/huggingface/notebooks/blob/main/lerobot/training-smolvla.ipynb#scrollTo=FMqvtVcThrdN

In [ ]:
import os, getpass
os.environ["WANDB_API_KEY"] = getpass.getpass("Paste your W&B API key: ")

import wandb
wandb.login() 

In [ ]:
import os, getpass
from huggingface_hub import login

token = getpass.getpass("Paste your HF token: ")
login(token=token)              # saves it for this machine, so later `!` commands find it
os.environ["HF_TOKEN"] = token

### Post-training SmolVLA using PEFT

In [ ]:
!pip uninstall -y torchao

In [ ]:
# !lerobot-train \
#  --resume=true \
#  --config_path=TomasAnderegg/pill_sorting_post_trained

!lerobot-train \
 --policy.path=lerobot/smolvla_base \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.optimizer_lr=1e-3 \
 --policy.device=cuda \
 --policy.scheduler_decay_lr=1e-4 \
 --policy.repo_id=TomasAnderegg/pill_sorting_post_trained_a100 \
 --num_workers=10  \
 --save_freq=3500 \
 --log_freq=100 \
 --policy.use_amp.mixed_precision=bf16  \
 --dataset.repo_id=TomasAnderegg/so101_pill_sorting_merged_v2 \
 --steps=10000 \
 --batch_size=64 \
 --peft.method_type=LORA \
 --peft.r=64 \
 --peft.lora_alpha=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true 

In [ ]:
!nvidia-smi


In [ ]:
!nproc

Calculation of the number of parameters used with LoRa:
1. Check the target used for LoRa: "(model\\.vlm_with_expert\\.lm_expert\\..*\\.(q|v)_proj|model\\.(state_proj|action_in_proj|action_out_proj|action_time_mlp_in|action_time_mlp_out))"
2. For each of them extract the features_in and features_out
3. Since LoRa adds 2 matrices A with size r x features_in and B with size features_out x r, we can calculate the total number for each adapted target.
4. Add everything

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")
print(policy)

In [ ]:
#Full post-training:
!lerobot-train \
 --policy.path=lerobot/smolvla_base \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.train_expert_only=false \
 --policy.freeze_vision_encoder=false \
 --policy.optimizer_lr=1e-4 \
 --policy.scheduler_decay_lr=2.5e-6 \
 --policy.scheduler_decay_steps=20000 \
 --policy.device=cuda \
 --policy.repo_id=TomasAnderegg/pill_sorting_full_ft_a100 \
 --job_name=pill_smolvla_full_ft \
 --num_workers=10 \
 --save_freq=5000 \
 --log_freq=100 \
 --policy.use_amp.mixed_precision=bf16  \
 --dataset.repo_id=TomasAnderegg/so101_pill_sorting_merged_v2 \
 --steps=20000 \
 --batch_size=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true

In [ ]:
## Fine tuning only the action expert
!lerobot-train \
 --policy.path=lerobot/smolvla_base \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.train_expert_only=true \
 --policy.freeze_vision_encoder=true \
 --policy.optimizer_lr=1e-4 \
 --policy.scheduler_decay_lr=2.5e-6 \
 --policy.scheduler_decay_steps=20000 \
 --policy.device=cuda \
 --policy.repo_id=TomasAnderegg/pill_sorting_expert_only_t4_better_data \
 --job_name=pill_smolvla_expert_only \
 --num_workers=10 \
 --save_freq=5000 \
 --log_freq=100 \
 --policy.use_amp.mixed_precision=bf16 \
 --dataset.repo_id=TomasAnderegg/so101_pill_sorting_v3 \
 --steps=20000 \
 --batch_size=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true

In [ ]:
## Fine tuning only the action expert using mixed precision: use accelerate 
!accelerate launch \
  --mixed_precision=bf16 \
    -m lerobot.scripts.lerobot_train \
 --policy.path=lerobot/smolvla_base \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.train_expert_only=true \
 --policy.freeze_vision_encoder=true \
 --policy.optimizer_lr=1e-4 \
 --policy.scheduler_decay_lr=2.5e-6 \
 --policy.scheduler_decay_steps=20000 \
 --policy.device=cuda \
 --policy.repo_id=TomasAnderegg/pill_sorting_expert_only_A100_big_data \
 --job_name=pill_smolvla_expert_only_v3 \
 --num_workers=10 \
 --save_freq=5000 \
 --log_freq=100 \
 --dataset.repo_id=TomasAnderegg/so101_pill_sorting_v3 \
 --steps=20000 \
 --batch_size=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true

In [ ]:
## Fine tuning only the action expert using mixed precision: use accelerate 
!accelerate launch \
  --mixed_precision=bf16 \
    -m lerobot.scripts.lerobot_train \
 --policy.path=TomasAnderegg/pill_sorting_expert_only_A100_big_data \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.train_expert_only=false \
 --policy.freeze_vision_encoder=true \
 --policy.optimizer_lr=3e-5 \
 --policy.scheduler_decay_lr=2.5e-6 \
 --policy.scheduler_decay_steps=20000 \
 --policy.device=cuda \
 --policy.repo_id=TomasAnderegg/pill_sorting_vlm_unfrozen_A100  \
 --job_name=pill_smolvla_vlm_unfrozen_A100 \
 --num_workers=10 \
 --save_freq=5000 \
 --log_freq=100 \
 --dataset.repo_id=TomasAnderegg/so101_pill_sorting_v3 \
 --steps=20000 \
 --batch_size=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true

### Dagger post training

In [ ]:
## Build the fine-tuning dataset: mix the original and the DAgger datasets, 50/50 ratio (by frames)

import json
from pathlib import Path
import numpy as np
import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

# DAgger dataset: its size sets the target
DAGGER_NAME = "rollout_pill_sorting_dagger_v3"
DAGGER_ROOT = Path("/mnt/c/Users/tjga9/Documents/Tomas/Projets/LeRobot/data") / DAGGER_NAME  # or your WSL copy
dagger = LeRobotDatasetMetadata(f"TomasAnderegg/{DAGGER_NAME}", root=DAGGER_ROOT)
target_frames = dagger.total_frames
print(f"DAgger: {dagger.total_episodes} episodes, {target_frames} frames")

# Original dataset
meta = LeRobotDatasetMetadata("TomasAnderegg/so101_pill_sorting_v3")
eps = meta.episodes
eps = eps.to_pandas() if hasattr(eps, "to_pandas") else pd.DataFrame(eps)
assert eps["tasks"].apply(len).max() == 1, "some episodes have several tasks"
eps["task"] = eps["tasks"].apply(lambda t: t[0])
lengths = eps.set_index("episode_index")["length"]

# Shuffle each task's episodes, then pick one per task in turn (round-robin)
# until the replay frames reach the DAgger frames → balanced tasks AND 50/50
rng = np.random.default_rng(0)
queues = {task: list(rng.permutation(g["episode_index"].to_numpy()))
          for task, g in eps.groupby("task")}

episode_list, n_frames = [], 0
while n_frames < target_frames and any(queues.values()):
    for task in rng.permutation(list(queues)):          # random task order each round
        if queues[task] and n_frames < target_frames:
            ep = int(queues[task].pop())
            episode_list.append(ep)
            n_frames += int(lengths[ep])
episode_list.sort()

print(f"Replay: {len(episode_list)} episodes, {n_frames} frames "
      f"({n_frames / (n_frames + target_frames):.0%} of the mix)")
print(eps.set_index("episode_index").loc[episode_list, "task"].value_counts())
print(json.dumps(episode_list))                         # paste into the split command

In [ ]:
## Split command: split the original dataset to mix with the dagger one. Selected according to the previous cell.

!lerobot-edit-dataset --repo_id=TomasAnderegg/so101_pill_sorting_v3 --operation.type=split --operation.splits '{"replay_subset" : [0, 15, 26, 30, 33, 36, 37, 40, 50, 56, 57, 58, 76, 82, 101, 103, 104, 106, 110, 111, 138, 149, 153, 157, 158, 162, 179, 183, 185, 192, 200, 211, 215, 216, 220, 224, 225, 227, 229, 250, 268, 283]}'


In [ ]:
!lerobot-edit-dataset \
  --repo_id=TomasAnderegg/rollout_pill_sorting_dagger_v3 \
  --new_repo_id=TomasAnderegg/pill_sorting_dagger_no_intervention_v3 \
  --operation.type=remove_feature \
  --operation.feature_names="['intervention']"

In [ ]:
!lerobot-edit-dataset \
  --new_repo_id=TomasAnderegg/pill_sorting_finetune_merged_v3 \
  --operation.type=merge \
  --operation.repo_ids="['TomasAnderegg/so101_pill_sorting_v3_replay_subset', 'TomasAnderegg/pill_sorting_dagger_no_intervention_v3']"

In [ ]:
import numpy as np
import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

metas = {
    "v3 (checkpoint)": LeRobotDatasetMetadata("TomasAnderegg/so101_pill_sorting_v3"),
    "replay subset":   LeRobotDatasetMetadata("TomasAnderegg/so101_pill_sorting_v3_replay_subset"),
    "dagger":          LeRobotDatasetMetadata("TomasAnderegg/pill_sorting_dagger_no_intervention_v3"),
}

def get(meta, key):
    s = meta.stats[key]
    return np.asarray(s["mean"], float), np.asarray(s["std"], float)

def pooled(ms, key):
    n = np.array([m.total_frames for m in ms], float)
    mus, sds = zip(*(get(m, key) for m in ms))
    mu, sd = np.stack(mus), np.stack(sds)
    w = n[:, None] / n.sum()                       # weight of each part by frame count
    mu_all = (w * mu).sum(0)
    sd_all = np.sqrt((w * (sd**2 + mu**2)).sum(0) - mu_all**2)
    return mu_all, sd_all

for name, m in metas.items():
    print(f"{name:16s} {m.total_episodes:4d} episodes  {m.total_frames:6d} frames")

for key in ["observation.state", "action"]:
    names = metas["v3 (checkpoint)"].features[key].get("names")
    if isinstance(names, dict):                    # some versions store {"motors": [...]}
        names = list(names.values())[0]

    rows = {}
    for name, m in metas.items():
        mu, sd = get(m, key)
        rows[(name, "mean")] = mu
        rows[(name, "std")] = sd

    mu_m, sd_m = pooled([metas["replay subset"], metas["dagger"]], key)
    mu_o, sd_o = get(metas["v3 (checkpoint)"], key)
    rows[("merged (est.)", "mean")] = mu_m
    rows[("merged (est.)", "std")] = sd_m
    rows[("shift vs v3", "Δmean / std_v3")] = (mu_m - mu_o) / sd_o
    rows[("shift vs v3", "std ratio")] = sd_m / sd_o

    print(f"\n=== {key} ===")
    display(pd.DataFrame(rows, index=names).round(3).T)

In [ ]:
## The DAgger data shifts the normalization stats, and the model was trained with v3's stats.
## So the merged dataset must use v3's stats; otherwise the same joint values would be normalized differently.
import shutil
from pathlib import Path
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

MERGED_NAME = "pill_sorting_finetune_merged_v3"
MERGED_ROOT = Path("/root/.cache/huggingface/lerobot/TomasAnderegg") / MERGED_NAME

v3     = LeRobotDatasetMetadata("TomasAnderegg/so101_pill_sorting_v3")
merged = LeRobotDatasetMetadata(f"TomasAnderegg/{MERGED_NAME}", root=MERGED_ROOT)

# Both must contain the same features, otherwise training will miss some stats
assert set(v3.stats) == set(merged.stats), (set(v3.stats) ^ set(merged.stats))

src = v3.root / "meta" / "stats.json"
dst = merged.root / "meta" / "stats.json"

# Back up the merged stats only if they haven't been swapped yet (safe to re-run)
if dst.read_bytes() != src.read_bytes():
    shutil.copy(dst, merged.root.parent / f"stats_{MERGED_NAME}_original.json")
shutil.copy(src, dst)                                                # v3 stats in place

# Check: every stat of state and action must match v3
reloaded = LeRobotDatasetMetadata(f"TomasAnderegg/{MERGED_NAME}", root=MERGED_ROOT)
for key in ["observation.state", "action"]:
    for stat in v3.stats[key]:
        same = np.allclose(np.asarray(reloaded.stats[key][stat], float),
                           np.asarray(v3.stats[key][stat], float))
        print(f"{key:18s} {stat:6s} {'OK' if same else 'DIFFERENT'}")

In [ ]:
## Sanity check before pushing to HF
import numpy as np
from huggingface_hub import HfApi

LOCAL  = MERGED_ROOT.parent
replay = LeRobotDatasetMetadata("TomasAnderegg/so101_pill_sorting_v3_replay_subset",
                                root=LOCAL / "so101_pill_sorting_v3_replay_subset")
dagger = LeRobotDatasetMetadata("TomasAnderegg/pill_sorting_dagger_no_intervention_v3",
                                root=LOCAL / "pill_sorting_dagger_no_intervention_v3")

print("merged folder:       ", merged.root)        # must be under /root/.cache/huggingface/lerobot/
print("already on Hub:      ", HfApi().repo_exists(f"TomasAnderegg/{MERGED_NAME}", repo_type="dataset"))
print("intervention removed:", "intervention" not in reloaded.features)

print(f"episodes: {reloaded.total_episodes} (expected {replay.total_episodes + dagger.total_episodes})")
print(f"frames:   {reloaded.total_frames} (expected {replay.total_frames + dagger.total_frames})")
print(f"DAgger share: {dagger.total_frames / reloaded.total_frames:.0%}")   # should be close to 50%

for key in ["observation.state", "action"]:
    for stat in v3.stats[key]:
        same = np.allclose(np.asarray(reloaded.stats[key][stat], float),
                           np.asarray(v3.stats[key][stat], float))
        print(f"{key:18s} {stat:6s} {'OK' if same else 'DIFFERENT'}")

In [ ]:
## Push to the Hub
from lerobot.datasets.lerobot_dataset import LeRobotDataset
LeRobotDataset(f"TomasAnderegg/{MERGED_NAME}", root=MERGED_ROOT).push_to_hub()

In [ ]:
## Fine-tuning only the action expert and language MOdel on the DAgger + replay mix (bf16 mixed precision, via accelerate)
!accelerate launch \
  --mixed_precision=bf16 \
    -m lerobot.scripts.lerobot_train \
 --policy.path=TomasAnderegg/pill_sorting_dagger_v3_and_subset \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.train_expert_only=false \
 --policy.freeze_vision_encoder=true \
 --policy.optimizer_lr=1e-5 \
 --policy.scheduler_warmup_steps=300 \
 --policy.scheduler_decay_lr=2.5e-6 \
 --policy.scheduler_decay_steps=2000 \
 --policy.device=cuda \
 --policy.repo_id=TomasAnderegg/pill_sorting_dagger_v3_and_subset_LM \
 --job_name=pill_smolvla_dagger_v3_A100_LM \
 --num_workers=10 \
 --save_freq=500 \
 --log_freq=100 \
 --dataset.repo_id=TomasAnderegg/pill_sorting_finetune_merged_v3 \
 --steps=2000 \
 --batch_size=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true

In [ ]:
import json
from pathlib import Path
from huggingface_hub import HfApi

BASE = "TomasAnderegg/pill_sorting_dagger_v3_and_subset_LM"

def run_repo(ckpt_dir):
    """Return the policy repo_id a training run pushes to (read from its saved config)."""
    steps = sorted(p for p in ckpt_dir.iterdir() if p.name.isdigit())
    if not steps:
        return None
    f = steps[-1] / "pretrained_model" / "train_config.json"
    return json.loads(f.read_text())["policy"].get("repo_id") if f.exists() else None

runs = list(Path("/content/outputs/train").glob("*/*/checkpoints"))
for r in runs:
    print(r.parent.name, "→", run_repo(r))

matches = [r for r in runs if run_repo(r) == BASE]
assert len(matches) == 1, f"found {len(matches)} runs for {BASE}; set RUN by hand from the list above"
RUN = matches[0]
print("Using:", RUN)

api = HfApi()
steps = sorted(p.name for p in RUN.iterdir() if p.name.isdigit())
print("steps:", steps)

for step in steps:
    repo = f"{BASE}_step{int(step)}"
    api.create_repo(repo, exist_ok=True, private=True)     # private: set False if you prefer
    api.upload_folder(folder_path=str(RUN / step / "pretrained_model"), repo_id=repo,
                      commit_message=f"Checkpoint at step {int(step)}")
    print("pushed", repo)

In [ ]:
!lerobot-edit-dataset \
  --repo_id=TomasAnderegg/rollout_pill_sorting_dagger_v3 \
  --new_repo_id=TomasAnderegg/pill_sorting_dagger_no_intervention_v3 \
  --operation.type=remove_feature \
  --operation.feature_names="['intervention']"

In [ ]:
## Merge the v3_dataset with the dagger v3 (pill_sorting_dagger_no_intervention_v3)
!lerobot-edit-dataset \
  --new_repo_id=TomasAnderegg/pill_sorting_big_datav3_dagger_60 \
  --operation.type=merge \
  --operation.repo_ids="['TomasAnderegg/so101_pill_sorting_v3', 'TomasAnderegg/pill_sorting_dagger_no_intervention_v3']"

In [ ]:
## Push to the Hub
from lerobot.datasets.lerobot_dataset import LeRobotDataset
LeRobotDataset(f"TomasAnderegg/pill_sorting_big_datav3_dagger_60", root="/home/tjga98/.cache/huggingface/lerobot/TomasAnderegg/pill_sorting_big_datav3_dagger_60").push_to_hub()

In [ ]:
#Full post-training with the 360 (v3 +dagger_v3) new episodes:
!accelerate launch \
  --mixed_precision=bf16 \
    -m lerobot.scripts.lerobot_train \
 --policy.path=lerobot/smolvla_base \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.train_expert_only=false \
 --policy.freeze_vision_encoder=false \
 --policy.optimizer_lr=1e-4 \
 --policy.scheduler_decay_lr=2.5e-6 \
 --policy.scheduler_decay_steps=20000 \
 --policy.device=cuda \
 --policy.repo_id=TomasAnderegg/pill_sorting_full_ft_a100_v3_dagger60 \
 --job_name=pill_smolvla_full_ft_v3_dagger60 \
 --num_workers=10 \
 --save_freq=2500 \
 --log_freq=100 \
 --dataset.repo_id=TomasAnderegg/pill_sorting_big_datav3_dagger_60 \
 --steps=20000 \
 --batch_size=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true

### Colour sorting dataset needs to be merged with big data v3 and dagger 60

In [ ]:
## Replay selection: episodes from so101_pill_sorting_v3, as many frames as the colour dataset (~50/50)
import json
from pathlib import Path
import numpy as np, pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

BIG    = "TomasAnderegg/pill_sorting_big_datav3_dagger_60"   # only for its stats.json (the checkpoint's stats)
SOURCE = "TomasAnderegg/so101_pill_sorting_v3"                # where the replay episodes come from
COLOUR = "TomasAnderegg/so101_pill_sorting_colour_v1"
LOCAL  = Path.home() / ".cache/huggingface/lerobot/TomasAnderegg"
RATIO  = 1.0          # replay frames / colour frames: 1.0 → 50 % colour

C = LeRobotDatasetMetadata(COLOUR).total_frames
target = int(C * RATIO)

eps = LeRobotDatasetMetadata(SOURCE).episodes
eps = eps.to_pandas() if hasattr(eps, "to_pandas") else pd.DataFrame(eps)
eps["task"] = eps["tasks"].apply(lambda t: t[0])
lengths = eps.set_index("episode_index")["length"]

rng = np.random.default_rng(0)
queues = {t: list(rng.permutation(g["episode_index"].to_numpy())) for t, g in eps.groupby("task")}
episode_list, n = [], 0
while n < target and any(queues.values()):
    for t in rng.permutation(list(queues)):
        if queues[t] and n < target:
            e = int(queues[t].pop()); episode_list.append(e); n += int(lengths[e])
episode_list.sort()
print(f"replay: {len(episode_list)} episodes, {n} frames → colour share {C / (C + n):.0%}")

In [ ]:
# Split and merge
import shutil

REPLAY = "so101_pill_sorting_v3_replay_colour"
MIX    = "pill_sorting_colour_mix_v1"
SPLITS = json.dumps({"replay_colour": episode_list})

shutil.rmtree(LOCAL / REPLAY, ignore_errors=True)   # clean re-runs
shutil.rmtree(LOCAL / MIX, ignore_errors=True)

!lerobot-edit-dataset --repo_id={SOURCE} --operation.type=split --operation.splits='{SPLITS}' 2>&1 | tail -20

r = LeRobotDatasetMetadata(f"TomasAnderegg/{REPLAY}", root=LOCAL / REPLAY)
print("replay:", r.total_episodes, "episodes,", r.total_frames, "frames")
assert r.total_episodes > 0, "the split failed: look at its output above"

!lerobot-edit-dataset --new_repo_id=TomasAnderegg/{MIX} --operation.type=merge --operation.repo_ids="['TomasAnderegg/{REPLAY}', '{COLOUR}']" 2>&1 | tail -20

In [ ]:
## The 10k checkpoint was trained with big_datav3_dagger_60's stats, so the mix must use the same stats;
## otherwise the same joint values would be normalized differently.
import shutil

ref    = LeRobotDatasetMetadata(BIG)                                     # stats the checkpoint learned with
merged = LeRobotDatasetMetadata(f"TomasAnderegg/{MIX}", root=LOCAL / MIX)
print("big has stats:", ref.stats is not None, "| mix has stats:", merged.stats is not None)

assert ref.stats is not None, f"no stats.json in {ref.root}"
assert set(ref.features) == set(merged.features), set(ref.features) ^ set(merged.features)

src, dst = ref.root / "meta" / "stats.json", merged.root / "meta" / "stats.json"
if dst.exists() and dst.read_bytes() != src.read_bytes():
    shutil.copy(dst, LOCAL / f"stats_{MIX}_original.json")               # backup, outside the dataset
shutil.copy(src, dst)
print("stats copied →", dst)

In [ ]:
## Sanity check before pushing to HF
import numpy as np
from huggingface_hub import HfApi

ref      = LeRobotDatasetMetadata(BIG)
reloaded = LeRobotDatasetMetadata(f"TomasAnderegg/{MIX}", root=LOCAL / MIX)      # was missing
replay   = LeRobotDatasetMetadata(f"TomasAnderegg/{REPLAY}", root=LOCAL / REPLAY)
colour   = LeRobotDatasetMetadata(COLOUR)

print("already on Hub:", HfApi().repo_exists(f"TomasAnderegg/{MIX}", repo_type="dataset"))
print(f"episodes: {reloaded.total_episodes} (expected {replay.total_episodes + colour.total_episodes})")
print(f"frames:   {reloaded.total_frames} (expected {replay.total_frames + colour.total_frames})")
print(f"colour share: {colour.total_frames / reloaded.total_frames:.0%}")      # should be close to 50 %

for key in ["observation.state", "action"]:
    for stat in ref.stats[key]:
        same = np.allclose(np.asarray(reloaded.stats[key][stat], float),
                           np.asarray(ref.stats[key][stat], float))
        print(f"{key:18s} {stat:6s} {'OK' if same else 'DIFFERENT'}")

In [ ]:
print("replay:", replay.total_episodes, "episodes,", replay.total_frames, "frames")
print("colour:", colour.total_episodes, "episodes,", colour.total_frames, "frames")
print("mix:   ", reloaded.total_episodes, "episodes,", reloaded.total_frames, "frames")

In [ ]:

from lerobot.datasets.lerobot_dataset import LeRobotDataset
LeRobotDataset(f"TomasAnderegg/{MIX}", root=LOCAL / MIX).push_to_hub()

In [ ]:
# Post-train again
## Colour fine-tune: from the final full-FT checkpoint, on the 50/50 colour mix, low LR
!accelerate launch --mixed_precision=bf16 -m lerobot.scripts.lerobot_train \
 --policy.path=TomasAnderegg/pill_sorting_full_ft_a100_v3_dagger60 \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.train_expert_only=false \
 --policy.freeze_vision_encoder=false \
 --policy.optimizer_lr=1e-5 \
 --policy.scheduler_warmup_steps=300 \
 --policy.scheduler_decay_lr=1e-6 \
 --policy.scheduler_decay_steps=4000 \
 --policy.device=cuda \
 --policy.repo_id=TomasAnderegg/pill_sorting_colour_ft_v1 \
 --job_name=pill_smolvla_colour_ft_v1 \
 --num_workers=10 \
 --save_freq=1000 \
 --log_freq=100 \
 --dataset.repo_id=TomasAnderegg/pill_sorting_colour_mix_v1 \
 --steps=4000 \
 --batch_size=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
  --save_checkpoint_to_hub=true 

In [4]:
## Option (a): same colour fine-tune, but without the wrong-start episodes
import json, shutil
from pathlib import Path
import numpy as np, pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

LOCAL  = Path.home() / ".cache/huggingface/lerobot/TomasAnderegg"
BIG    = "TomasAnderegg/pill_sorting_big_datav3_dagger_60"   # the starting checkpoint was trained on it (its stats)
SOURCE = "TomasAnderegg/so101_pill_sorting_v3"               # replay episodes come from here
COLOUR = "so101_pill_sorting_colour_v1_direct"               # colour dataset, direct episodes only
REPLAY = "so101_pill_sorting_v3_replay_direct"               # = v3 name + "_" + split name
MIX    = "pill_sorting_colour_direct_mix_v1"
PLAN   = "/mnt/c/Users/tjga9/Documents/Tomas/Projets/LeRobot/recording_plan.csv"   # next to the recording script

In [5]:
## Episodes that start above a wrong pill (from the recording plan)
plan  = pd.read_csv(PLAN, encoding="utf-8-sig")
wrong = plan.loc[plan.Start == "wrong", "Episode"].tolist()
assert wrong == [i for i in range(96) if i % 4 >= 2]     # per colour: direct, direct, wrong, wrong
print(len(wrong), "wrong-start episodes:", wrong)

48 wrong-start episodes: [2, 3, 6, 7, 10, 11, 14, 15, 18, 19, 22, 23, 26, 27, 30, 31, 34, 35, 38, 39, 42, 43, 46, 47, 50, 51, 54, 55, 58, 59, 62, 63, 66, 67, 70, 71, 74, 75, 78, 79, 82, 83, 86, 87, 90, 91, 94, 95]


In [15]:
## Remove them → new dataset with the 48 direct episodes only
shutil.rmtree(LOCAL / COLOUR, ignore_errors=True)
!lerobot-edit-dataset --repo_id=TomasAnderegg/so101_pill_sorting_colour_v1 --new_repo_id=TomasAnderegg/{COLOUR} --operation.type=delete_episodes --operation.episode_indices="{wrong}" 2>&1 | tail -5

colour = LeRobotDatasetMetadata(f"TomasAnderegg/{COLOUR}", root=LOCAL / COLOUR)
print(colour.total_episodes, "episodes,", colour.total_frames, "frames")    # expected: 48 episodes

Processing episodes metadata: 100%|██████████| 48/48 [00:00<00:00, 74.50it/s]
INFO 2026-11-01 14:19:52 et_tools.py:975 Aggregating statistics for 48 episodes
INFO 2026-11-01 14:19:53 et_tools.py:173 Created new dataset with 48 episodes
INFO 2026-11-01 14:19:53 _dataset.py:463 Dataset saved to /home/tjga98/.cache/huggingface/lerobot/TomasAnderegg/so101_pill_sorting_colour_v1_direct
INFO 2026-11-01 14:19:53 _dataset.py:464 Episodes: 48, Frames: 17755
48 episodes, 17755 frames


In [16]:
## Replay selection: v3 episodes, as many frames as the colour dataset (50/50), balanced over tasks
target = colour.total_frames
eps = LeRobotDatasetMetadata(SOURCE).episodes
eps = eps.to_pandas() if hasattr(eps, "to_pandas") else pd.DataFrame(eps)
eps["task"] = eps["tasks"].apply(lambda t: t[0])
lengths = eps.set_index("episode_index")["length"]

rng = np.random.default_rng(0)
queues = {t: list(rng.permutation(g["episode_index"].to_numpy())) for t, g in eps.groupby("task")}
episode_list, n = [], 0
while n < target and any(queues.values()):
    for t in rng.permutation(list(queues)):
        if queues[t] and n < target:
            e = int(queues[t].pop()); episode_list.append(e); n += int(lengths[e])
episode_list.sort()
print(f"replay: {len(episode_list)} episodes, {n} frames → colour share {target / (target + n):.0%}")

Fetching 67 files: 100%|██████████| 67/67 [00:00<00:00, 1011.74it/s]

replay: 20 episodes, 19198 frames → colour share 48%


In [17]:
## Split: copy the selected v3 episodes into a new dataset
SPLITS = json.dumps({"replay_direct": episode_list})
shutil.rmtree(LOCAL / REPLAY, ignore_errors=True)
!lerobot-edit-dataset --repo_id={SOURCE} --operation.type=split --operation.splits='{SPLITS}' 2>&1 | tail -5

replay = LeRobotDatasetMetadata(f"TomasAnderegg/{REPLAY}", root=LOCAL / REPLAY)
print(replay.total_episodes, "episodes,", replay.total_frames, "frames")

Processing episodes metadata: 100%|██████████| 20/20 [00:00<00:00, 28.64it/s]
INFO 2026-11-01 14:20:04 et_tools.py:975 Aggregating statistics for 20 episodes
INFO 2026-11-01 14:20:04 _dataset.py:496 replay_direct: 20 episodes, 19198 frames
20 episodes, 19198 frames


In [18]:
## Merge: replay + colour → the mix
shutil.rmtree(LOCAL / MIX, ignore_errors=True)
!lerobot-edit-dataset --new_repo_id=TomasAnderegg/{MIX} --operation.type=merge --operation.repo_ids="['TomasAnderegg/{REPLAY}', 'TomasAnderegg/{COLOUR}']" 2>&1 | tail -5

INFO 2026-11-01 14:20:40 ggregate.py:816 write info
INFO 2026-11-01 14:20:40 ggregate.py:823 write stats
INFO 2026-11-01 14:20:40 ggregate.py:449 Aggregation complete.
INFO 2026-11-01 14:20:41 _dataset.py:541 Merged dataset saved to /home/tjga98/.cache/huggingface/lerobot/TomasAnderegg/pill_sorting_colour_direct_mix_v1
INFO 2026-11-01 14:20:41 _dataset.py:542 Episodes: 68, Frames: 36953


In [19]:
## The starting checkpoint was trained with big_datav3_dagger_60's stats, so the mix must use the same ones;
## otherwise the same joint values would be normalized differently.
ref = LeRobotDatasetMetadata(BIG)
shutil.copy(ref.root / "meta" / "stats.json", LOCAL / MIX / "meta" / "stats.json")

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 649.95it/s]


PosixPath('/home/tjga98/.cache/huggingface/lerobot/TomasAnderegg/pill_sorting_colour_direct_mix_v1/meta/stats.json')

In [20]:
## Sanity check before pushing
mix = LeRobotDatasetMetadata(f"TomasAnderegg/{MIX}", root=LOCAL / MIX)
print(f"episodes: {mix.total_episodes} (expected {replay.total_episodes + colour.total_episodes})")
print(f"frames:   {mix.total_frames} (expected {replay.total_frames + colour.total_frames})")
print(f"colour share: {colour.total_frames / mix.total_frames:.0%}")      # should be close to 50 %

for key in ["observation.state", "action"]:
    for stat in ref.stats[key]:
        same = np.allclose(np.asarray(mix.stats[key][stat], float), np.asarray(ref.stats[key][stat], float))
        print(f"{key:18s} {stat:6s} {'OK' if same else 'DIFFERENT'}")

episodes: 68 (expected 68)
frames:   36953 (expected 36953)
colour share: 48%
observation.state  min    OK
observation.state  max    OK
observation.state  mean   OK
observation.state  std    OK
observation.state  count  OK
observation.state  q01    OK
observation.state  q10    OK
observation.state  q50    OK
observation.state  q90    OK
observation.state  q99    OK
action             min    OK
action             max    OK
action             mean   OK
action             std    OK
action             count  OK
action             q01    OK
action             q10    OK
action             q50    OK
action             q90    OK
action             q99    OK


In [21]:
## Push to the Hub
LeRobotDataset(f"TomasAnderegg/{MIX}", root=LOCAL / MIX).push_to_hub()

No files have been modified since last commit. Skipping to prevent empty commit.


In [ ]:
## Colour fine-tune (direct episodes only): from the final full-FT checkpoint, on the 50/50 mix, low LR
!accelerate launch --mixed_precision=bf16 -m lerobot.scripts.lerobot_train \
 --policy.path=TomasAnderegg/pill_sorting_full_ft_a100_v3_dagger60 \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.train_expert_only=false \
 --policy.freeze_vision_encoder=false \
 --policy.optimizer_lr=1e-5 \
 --policy.scheduler_warmup_steps=300 \
 --policy.scheduler_decay_lr=1e-6 \
 --policy.scheduler_decay_steps=4000 \
 --policy.device=cuda \
 --policy.repo_id=TomasAnderegg/pill_sorting_colour_direct_ft_v1 \
 --job_name=pill_smolvla_colour_direct_ft_v1 \
 --num_workers=10 \
 --save_freq=1000 \
 --log_freq=100 \
 --dataset.repo_id=TomasAnderegg/pill_sorting_colour_direct_mix_v1 \
 --steps=4000 \
 --batch_size=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true